# Explore World Bank Data


## Pre-requisites

1. Run the `prepare_world_bank_data.ipynb` to download data via the [Data 360 API](https://data360.worldbank.org/en/api) and prepare the data in a DuckDB database.

In [ ]:
import polars as pl
import duckdb
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
DUCKDB_PATH = Path("../../databases") / "world_bank.db"
assert DUCKDB_PATH.exists(), f"Database file not found at {DUCKDB_PATH}"

In [6]:
with duckdb.connect(str(DUCKDB_PATH), read_only=True) as db:
    countries = db.sql("SELECT * FROM silver.countries").pl()

In [7]:
countries["region"].value_counts()

region,count
str,u32
"""South Asia""",6
"""Middle East, North Africa, Afg…",23
"""Europe & Central Asia""",58
"""North America""",3
"""East Asia & Pacific""",37
"""Sub-Saharan Africa """,48
"""Latin America & Caribbean """,42


In [9]:
countries = (
    countries
    .select(["country_code", "iso2_code", "country_name", "region", "capital_city", "longitude", "latitude"])  # Select only relevant columns.
    .sort(["country_name"])  # Sort by country name.
)
countries

country_code,iso2_code,country_name,region,capital_city,longitude,latitude
str,str,str,str,str,str,str
"""AFG""","""AF""","""Afghanistan""","""Middle East, North Africa, Afg…","""Kabul""","""69.1761""","""34.5228"""
"""ALB""","""AL""","""Albania""","""Europe & Central Asia""","""Tirane""","""19.8172""","""41.3317"""
"""DZA""","""DZ""","""Algeria""","""Middle East, North Africa, Afg…","""Algiers""","""3.05097""","""36.7397"""
"""ASM""","""AS""","""American Samoa""","""East Asia & Pacific""","""Pago Pago""","""-170.691""","""-14.2846"""
"""AND""","""AD""","""Andorra""","""Europe & Central Asia""","""Andorra la Vella""","""1.5218""","""42.5075"""
…,…,…,…,…,…,…
"""VIR""","""VI""","""Virgin Islands (U.S.)""","""Latin America & Caribbean ""","""Charlotte Amalie""","""-64.8963""","""18.3358"""
"""PSE""","""PS""","""West Bank and Gaza""","""Middle East, North Africa, Afg…","""""","""""",""""""
"""YEM""","""YE""","""Yemen, Rep.""","""Middle East, North Africa, Afg…","""Sana'a""","""44.2075""","""15.352"""


## Complex transformation, leveraging DuckDB and Polars

When you pair DuckDB with Polars, you get the best of both worlds:
*   **High-Performance SQL:** Use DuckDB's fast SQL engine to perform initial filtering, aggregation, and data manipulation at the database level.
*   **Expressive DataFrame API:** Load the results directly into a Polars DataFrame to leverage its powerful and expressive API for more complex transformations and analysis.

In this example, we are first going to use a DuckDB prepare the data through a more complex query which joins two tables and filters the data through a `WHERE` clause.

In [29]:
with duckdb.connect(str(DUCKDB_PATH), read_only=True) as db:
    
    duckdb_query_results = db.sql(
        """
        SELECT d.country_code, c.country_name, c.region, d.year, d.indicator_code, d.value, c.longitude, c.latitude
        FROM silver.data d
        JOIN silver.countries c ON d.country_code = c.country_code
        WHERE d.indicator_code IN ('WB_WDI_SP_POP_TOTL', 'WB_WDI_SP_DYN_LE00_IN', 'WB_WDI_NY_GDP_PCAP_CD')
        """
        ).pl()

duckdb_query_results.head(3)

country_code,country_name,region,year,indicator_code,value,longitude,latitude
str,str,str,i64,str,f64,str,str
"""VNM""","""Viet Nam""","""East Asia & Pacific""",1999,"""WB_WDI_NY_GDP_PCAP_CD""",375.994456,"""105.825""","""21.0069"""
"""VNM""","""Viet Nam""","""East Asia & Pacific""",2001,"""WB_WDI_NY_GDP_PCAP_CD""",419.205678,"""105.825""","""21.0069"""
"""VNM""","""Viet Nam""","""East Asia & Pacific""",2008,"""WB_WDI_NY_GDP_PCAP_CD""",1163.831958,"""105.825""","""21.0069"""


Next we are going to use a [`polars.DataFrame.pivot`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.pivot.html) operation to transform the shape of the dataframe and get the data ready to plot on a chart.

In [32]:
world_wealth_and_health = (
    duckdb_query_results
    .pivot(
        on=["indicator_code"],
        index=["country_code", "country_name", "region", "year", "longitude", "latitude"],
        values="value"
        )
    .rename(
        {
            "WB_WDI_NY_GDP_PCAP_CD": "gdp_usd_per_capita",
            "WB_WDI_SP_DYN_LE00_IN": "life_expectancy",
            "WB_WDI_SP_POP_TOTL": "population"
        }
        )
    .sort(["country_code", "year"])
    .with_columns(
        pl.col("gdp_usd_per_capita").forward_fill().over("country_code"),
        pl.col("life_expectancy").forward_fill().over("country_code"),
        pl.col("population").forward_fill().over("country_code"),
    )
    .drop_nulls(subset=["gdp_usd_per_capita", "life_expectancy", "population"])
    .with_columns(
        [
            (pl.col("population") / 1000000).round(2).alias("population_in_millions"),
        ]
    )
)

In [33]:
world_wealth_and_health.head(3)

country_code,country_name,region,year,longitude,latitude,gdp_usd_per_capita,life_expectancy,population,population_in_millions
str,str,str,i64,str,str,f64,f64,f64,f64
"""ALB""","""Albania""","""Europe & Central Asia""",1980,"""19.8172""","""41.3317""",590.607738,69.903,2.671997e6,2.67
"""ALB""","""Albania""","""Europe & Central Asia""",1981,"""19.8172""","""41.3317""",663.294208,70.145,2.726056e6,2.73
"""ALB""","""Albania""","""Europe & Central Asia""",1982,"""19.8172""","""41.3317""",668.454504,70.425,2.784278e6,2.78


In [ ]:
# Display the results on a geographical scatter plot.
fig = px.scatter_geo(
    world_wealth_and_health,
    lat="latitude",
    lon="longitude",
    hover_name="country_name",
    color="region",           # Color points by region
    projection="natural earth",
    title="World Bank Data - Country Locations"
)

fig.show()

Finally we chart the results to show the snail trail of each country over time on a two dimensional scatter chart.

In [28]:
fig = px.scatter(
    world_wealth_and_health,
    x="gdp_usd_per_capita",
    y="life_expectancy",
    animation_frame="year",
    animation_group="country_name",
    size="population_in_millions",
    color="region",
    text="country_code",
    hover_name="country_name",
    log_x=True,
    size_max=120,
    range_x=[100, 100000],
    range_y=[25, 90],
    title="World Wealth and Health Over Time",
    labels={
        "gdp_usd_per_capita": "Wealth (GDP Per Capita in USD)",
        "life_expectancy": "Health (Life Expectancy in Years)"
    }
)
fig.show()

## Analysis of Population

In [24]:
uk_population = (
    world_wealth_and_health
    # Filter in one step based on country name and year
    .filter(pl.col("country_name") == "United Kingdom")  # type: ignore
    # Add a new column for population in millions
    .with_columns((pl.col("population") / 1000000).alias("population_in_millions"))  
    .sort("year", descending=False)
    .with_columns(
        (((pl.col("population_in_millions") - pl.col("population_in_millions").shift(1))) / pl.col("population_in_millions").shift(1) * 100)
        .alias("population_change_percentage")  # Add new column for population change percentage
    )
    .with_columns(
        pl.when(pl.col("population_change_percentage") > 0)
        .then(pl.lit("green"))
        .otherwise(pl.lit("red"))
        .alias("color")  # Add new column for color based on population change
    )
    .select(["year", "population_in_millions", "population_change_percentage", "color"])
)

In [25]:
# Create subplots with 2 rows and 1 column
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,  # Space between charts
    row_heights=[0.7, 0.3],  # 70% height for main chart, 30% for bar chart
    subplot_titles=("Total Population (Millions)", "Year-on-Year Growth (%)")
)

# Top Chart: Absolute Population (Line + Markers)
fig.add_trace(
    go.Scatter(
        x=uk_population["year"],
        y=uk_population["population_in_millions"],
        mode="lines+markers",
        name="Population",
        line=dict(width=3)
    ),
    row=1, col=1
)

# Bottom Chart: Percentage Change (Bar)
fig.add_trace(
    go.Bar(
        x=uk_population["year"],
        y=uk_population["population_change_percentage"],
        marker_color=uk_population["color"],  # Use the calculated red/green column
        name="Change %"
    ),
    row=2, col=1
)

# Update layout configuration
fig.update_layout(
    title_text="United Kingdom Population Analysis (Last 50 Years)",
    showlegend=False
)

fig.show()